# Case 44 warm-up: Q-learning without neural networks

This finite-state warm-up is based on the publisher’s Catch environment. It does not reproduce the original neural-network or financial A2C cases. We observe `(fruit_row, fruit_column, basket_position)` directly and use a Q table instead of image inputs and a neural network. Success in this finite toy environment cannot be extrapolated to financial performance.

Run from this notebook’s `warmup/` directory. Only the Python standard library is needed.


In [1]:
from pathlib import Path
from tabular_catch import step, target, run, check_transition_contract
print("transition:", step((0, 3, 2), 2))
print("terminal target:", target(-1, True, [100, 100, 100], 0.95))


transition: ((1, 3, 3), 0, False)
terminal target: -1


## Q update

`Q(s,a) ← Q(s,a) + α [r + γ max Q(s′,a′) − Q(s,a)]`; for a terminal transition, the target is only `r`.

Actions 0/1/2 mean left/stay/right. The fruit drops one row per step, and the environment clips the basket position to the boundaries. Catching the fruit yields +1; otherwise the reward is −1. Boundary constraints are enforced by the environment, not learned through rewards.


In [2]:
old_q, alpha, gamma = 0.4, 0.25, 0.95
new_q = old_q + alpha * (target(0, False, [0.2, 0.8, 0.1], gamma) - old_q)
print("one Q update:", round(new_q, 4))
print("finite checks:", check_transition_contract())


one Q update: 0.49
finite checks: 1176


## Small experiment: explicit exploration rate

Hold the grid, learning rate, discount, and training episodes constant. Compare epsilon=0 with 0.2 across five fixed seeds. Training breaks ties between equal Q values randomly, so epsilon=0 does not eliminate all randomness.

Evaluate fixed policies over every initial state in the original environment’s distribution support, comparing against stay and random-action baselines. Evaluation does not update the Q table, but it uses the same MDP and is not evidence of generalization to unseen market data.


In [3]:
run(Path("results"))


Finite transition checks passed: 1176
tabular_q, epsilon=0.0: mean success=1.000, min=1.000, max=1.000
tabular_q, epsilon=0.2: mean success=1.000, min=1.000, max=1.000
stay, epsilon=: mean success=0.429, min=0.429, max=0.429
random, epsilon=: mean success=0.440, min=0.400, max=0.514


## Results and discussion

Full metrics are in `results/metrics.csv`; training returns, evaluation trajectories, and configuration are also saved. These results concern only this finite environment. The comparison does not establish that either exploration rate is generally better.

Explain independently: why is this toy environment suitable for a lookup table? Why does increasing reward alone not prove a constraint? Which assumptions change when neural networks or financial data are introduced?

Next compare `../upstream/7.1 Q-Learning.ipynb`, then study the financial A2C data and code. The formal presentation still requires reproduction and extension of the instructor-assigned case.
